In [52]:
import rasterio
from rasterio.warp import transform
import sys
!{sys.executable} -m pip install ipynb

import numpy as np
import torch
from torch import nn, distributions
from scipy.spatial.distance import cdist
from scipy.stats import norm
import matplotlib.pyplot as plt

import random

torch.set_default_tensor_type(torch.FloatTensor)

In [53]:
vrt_path = "C:/Users/karlj/Desktop/nasadem_all/nasadem.vrt"     
#example query
points_lonlat = [(12.5683, 55.6761), (12.60, 55.70)]  # (lon, lat)

In [54]:
def query(vrt_path, points_lonlat):
    with rasterio.open(vrt_path) as src:
        # Reproject input lon/lat -> dataset CRS if needed
        if src.crs and src.crs.to_string() != "EPSG:4326":
            lons, lats = zip(*points_lonlat)
            xs, ys = transform("EPSG:4326", src.crs, lons, lats)
            pts = list(zip(xs, ys))
        else:
            pts = points_lonlat

        # Sample band 1
        vals = [v[0] for v in src.sample(pts)]
        return vals

In [55]:
def squared_exponential_kernel(x, y, lengthscale, variance):
    '''
    Function that computes the covariance matrix using a squared-exponential kernel
    Handles both 1D and multi-dimensional inputs using Euclidean distance
    '''
    # Ensure inputs are 2D arrays (N, D) where D is the dimension
    x = np.atleast_2d(x)
    y = np.atleast_2d(y)
    
    # pair-wise squared Euclidean distances, size: NxM
    sqdist = cdist(x, y, 'sqeuclidean')
    # compute the kernel
    cov_matrix = variance * np.exp(-0.5 * sqdist * (1/lengthscale**2))  # NxM
    return cov_matrix


def fit_predictive_GP(X, y, Xtest, lengthscale, kernel_variance, noise_variance):
    '''
    Function that fits the Gaussian Process. It returns the predictive mean function and
    the predictive covariance function. It follows step by step the algorithm on the lecture
    notes. Handles multi-dimensional inputs.
    '''
    X = np.atleast_2d(X)
    y = np.array(y).flatten()
    Xtest = np.atleast_2d(Xtest)
    
    K = squared_exponential_kernel(X, X, lengthscale, kernel_variance)
    L = np.linalg.cholesky(K + noise_variance * np.eye(len(X)))

    # compute the mean at our test points.
    Ks = squared_exponential_kernel(X, Xtest, lengthscale, kernel_variance)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    mu = Ks.T @ alpha

    v = np.linalg.solve(L, Ks)
    # compute the variance at our test points.
    Kss = squared_exponential_kernel(Xtest, Xtest, lengthscale, kernel_variance)
    covariance = Kss - (v.T @ v)
    
    return mu, covariance


# I am using PyTorch to define the optimization function, it can be done in different other ways.
# It is not the best way to implement it, I suppose
def optimize_GP_hyperparams(Xtrain, ytrain, optimization_steps, learning_rate, mean_prior, prior_std):
    '''
    Methods that run the optimization of the hyperparams of our GP. We will use
    Gradient Descent because it takes too much time to run grid search at each step
    of bayesian optimization. We use a different definition of the kernel to make the
    optimization more stable. Handles multi-dimensional inputs.

    :param X: training set points
    :param y: training targets
    :return: values for lengthscale, output_var, noise_var that maximize the log-likelihood
    '''
    
    # we are re-defining the kernel because we need it in PyTorch
    def squared_exponential_kernel_torch(x, y, _lambda, variance):
        # x: (N, D), y: (M, D) where D is the dimensionality
        # Compute squared Euclidean distances between all pairs
        x_expanded = x.unsqueeze(1)  # (N, 1, D)
        y_expanded = y.unsqueeze(0)  # (1, M, D)
        sqdist = torch.sum((x_expanded - y_expanded) ** 2, dim=2)  # (N, M)
        k = variance * torch.exp(-0.5 * sqdist * (1/_lambda**2))
        return k

    # Convert to proper shape - ensure 2D array
    X = np.atleast_2d(Xtrain)
    y = np.array(ytrain).reshape(-1, 1)
    N = len(X)

    # transform our training set in Tensor
    Xtrain_tensor = torch.from_numpy(X).float()
    ytrain_tensor = torch.from_numpy(y).float()
    # we should define our hyperparameters as torch parameters where we keep track of
    # the operations to get the gradients from them
    _lambda = nn.Parameter(torch.tensor(1.), requires_grad=True)
    output_variance = nn.Parameter(torch.tensor(1.), requires_grad=True)
    noise_variance = nn.Parameter(torch.tensor(.5), requires_grad=True)

    # we use Adam as optimizer
    optim = torch.optim.Adam([_lambda, output_variance, noise_variance], lr=learning_rate)

    # optimization loop using the log-likelihood that involves the cholesky decomposition 
    nlls = []
    lambdas = []
    output_variances = []
    noise_variances = []
    iterations = optimization_steps
    for i in range(iterations):
        assert noise_variance >= 0, f"ouch! {i, noise_variance}"
        optim.zero_grad()
        K = squared_exponential_kernel_torch(Xtrain_tensor, Xtrain_tensor, _lambda,
                                                output_variance) + noise_variance * torch.eye(N)
        
        L = torch.linalg.cholesky(K)
        _alpha_temp = torch.linalg.solve_triangular(L, ytrain_tensor, upper=False)
        _alpha = torch.linalg.solve_triangular(L.t(), _alpha_temp, upper=True)
        nll = N / 2 * torch.log(torch.tensor(2 * np.pi)) + 0.5 * torch.matmul(ytrain_tensor.transpose(0, 1),
                                                                              _alpha) + torch.sum(torch.log(torch.diag(L)))

        # we have to add the log-likelihood of the prior
        norm = distributions.Normal(loc=mean_prior, scale=prior_std)
        prior_negloglike = torch.log(_lambda) - norm.log_prob(_lambda)

        nll += 0.9 * prior_negloglike
        nll.backward()

        nlls.append(nll.item())
        lambdas.append(_lambda.item())
        output_variances.append(output_variance.item())
        noise_variances.append(noise_variance.item())
        optim.step()

        # projected in the constraints (lengthscale and output variance should be positive)
        for p in [_lambda, output_variance]:
            p.data.clamp_(min=0.0000001)

        noise_variance.data.clamp_(min=1e-5, max=0.05)

        
    return _lambda.item(), output_variance.item(), noise_variance.item()

In [56]:
def expected_improvement(current_best, mean, std, xi):
    '''
    It implements the following function:

            | (mu - mu^+ - xi) Phi(Z) + sigma phi(Z) if sigma > 0
    EI(x) = |
            | 0                                       if sigma = 0

            where Phi is the CDF and phi the PDF of the normal distribution
            and
            Z = (mu - mu^+ - xi) / sigma

    :param current_best: this is the current max of the unknown function: mu^+
    :param mean: this is the mean function from the GP over the considered set of points
    :param std: this is the std function from the GP over the considered set of points
    :param xi: small value added to avoid corner case
    :return: the value of this acquisition function for all the points
    '''

    # start by computing the Z as we did in the probability of improvement function
    # to avoid division by 0, add a small term eg. np.spacing(1e6) to the denominator
    Z = (mean - current_best - xi) / (std + 1e-9) # or Z = (mean - current_best - eps) / (std + np.spacing(1e6))
    # now we have to compute the output only for the terms that have their std > 0
    EI = (mean - current_best - xi) * norm.cdf(Z) + std * norm.pdf(Z)
    #EI[std == 0] = 0
    
    return EI


In [61]:

# Define the region bounds
lat = (46, 49) 
long = (9, 17)


# Create a 2D grid of candidate points
N_grid = 50  # Grid resolution (50x50 = 2500 points)
lon_range = np.linspace(long[0], long[1], N_grid)
lat_range = np.linspace(lat[0], lat[1], N_grid)
lon_grid, lat_grid = np.meshgrid(lon_range, lat_range)
X = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # Shape: (2500, 2)

print(f"Candidate grid shape: {X.shape} (each point is [lon, lat])")


# prior over the lengthscale
prior_mean = 400 
prior_std = 1000

# array for the points x' and respective f(x') that we sample using the acquisition function
X_sample = []
y_sample = []

# number of iterations
T = 3

# trade-off hyperparams
eps = 5

#number of initial samples
n_start = 10

for i in range(n_start):
    lo = random.uniform(long[0],long[1])
    la = random.uniform(lat[0], lat[1])
    X_sample.append((lo,la))
    y_sample(vrt_path,[(lo,la)])
    
# bayesian optimization loop
# compute the current best 
current_best = np.max(y_sample)

# loop
acquisition = 'improvement'

for t in range(len(X_sample), len(X_sample) + T):
    print(f"\n{'='*60}")
    print(f"Iteration {t+1}/{len(X_sample) + T}")
    print(f"{'='*60}")
    
    ## we should optimize the GP hyperparameters before fitting the final GP and computing the acquisition
    ## functions
    lengthscale, output_variance, noise_variance = optimize_GP_hyperparams(X_sample, y_sample, 500, 5e-4, prior_mean, prior_std)
    print(f"GP Hyperparameters: lengthscale={lengthscale:.3f}, output_var={output_variance:.3f}, noise_var={noise_variance:.5f}")
    
    # we have to fit the GP using the X_sample, Y_sample and our training points

    lon = random.uniform(long[0],long[1])
    lat = random.uniform(lat[0], lat[1])
    X = (lon,lat)
    X_sample.append(X)
    mu, covariance = fit_predictive_GP(X_sample, y_sample, X, lengthscale, output_variance, noise_variance)
    # get the standard deviation
    std = np.sqrt(np.diag(covariance))

    ## calculate acquisition values for your acquisition function of choice
    ## now we have to use the acquisition function

    if acquisition == 'improvement':
        acquisition_values = expected_improvement(current_best, mu, std, eps)
    
    # we have to find the xt that maximizes this acquisition function
    best_idx = np.argmax(acquisition_values)
    xt = tuple(X[best_idx])  # Convert to (lon, lat) tuple


    y_new = query(vrt_path, [xt])[0]
    
    # we compute the value of xt and we add xt and f(xt) in X_samples and y_samples
    X_sample.append(xt)
    y_sample.append(y_new)

    # update current best
    current_best = np.max(y_sample)

    ## now we should create the plots
    plt.figure(figsize=(16, 10))
    
    # Reshape for 2D plotting
    mu_grid = mu.reshape(N_grid, N_grid)
    std_grid = std.reshape(N_grid, N_grid)
    acq_grid = acquisition_values.reshape(N_grid, N_grid)
    
    # Plot 1: Mean function
    plt.subplot(2, 3, 1)
    plt.title(f'Mean Function (t={t+1})')
    im1 = plt.contourf(lon_grid, lat_grid, mu_grid, levels=20, cmap='viridis')
    plt.colorbar(im1, label='Predicted Elevation (m)')
    plt.plot([x[0] for x in X_sample], [x[1] for x in X_sample], 'ro', markersize=8, label='Sampled')
    plt.plot(xt[0], xt[1], 'y*', markersize=20, label='Next sample')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 2: Std function
    plt.subplot(2, 3, 2)
    plt.title(f'Std Function (t={t+1})')
    im2 = plt.contourf(lon_grid, lat_grid, std_grid, levels=20, cmap='plasma')
    plt.colorbar(im2, label='Std')
    plt.plot([x[0] for x in X_sample], [x[1] for x in X_sample], 'ro', markersize=8)
    plt.plot(xt[0], xt[1], 'y*', markersize=20)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True, alpha=0.3)
    
    # Plot 3: Acquisition function
    plt.subplot(2, 3, 3)
    plt.title(f'Acquisition Function (t={t+1})')
    im3 = plt.contourf(lon_grid, lat_grid, acq_grid, levels=20, cmap='coolwarm')
    plt.colorbar(im3, label='Acquisition Value')
    plt.plot([x[0] for x in X_sample], [x[1] for x in X_sample], 'ro', markersize=8)
    plt.plot(xt[0], xt[1], 'y*', markersize=20)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True, alpha=0.3)
    
    # Plot 4: Sampled points progression
    plt.subplot(2, 3, 4)
    plt.title('Sampled Points Progression')
    scatter = plt.scatter([x[0] for x in X_sample], [x[1] for x in X_sample], 
                         c=y_sample, cmap='viridis', s=150, edgecolors='black', linewidths=2)
    plt.plot([x[0] for x in X_sample], [x[1] for x in X_sample], 'r:', alpha=0.5)
    for i, (x, y_val) in enumerate(zip(X_sample, y_sample)):
        plt.text(x[0], x[1], str(i+1), ha='center', va='center', 
                color='white', fontweight='bold', fontsize=10)
    plt.colorbar(scatter, label='Elevation (m)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True, alpha=0.3)
    
    # Plot 5: Mean with uncertainty bands (1D slice)
    plt.subplot(2, 3, 5)
    plt.title('Mean ± Std (Longitude slice at center lat)')
    center_lat_idx = N_grid // 2
    lon_slice = lon_range
    mu_slice = mu_grid[center_lat_idx, :]
    std_slice = std_grid[center_lat_idx, :]
    plt.fill_between(lon_slice, mu_slice - std_slice, mu_slice + std_slice, 
                     color='lightblue', alpha=0.5, label=r'$\pm 1\sigma$')
    plt.plot(lon_slice, mu_slice, 'b-', label='Mean')
    # Plot sampled points on this slice
    for x_samp, y_samp in zip(X_sample, y_sample):
        if abs(x_samp[1] - lat_range[center_lat_idx]) < (lat[1] - lat[0]) / N_grid:
            plt.plot(x_samp[0], y_samp, 'ro', markersize=8)
    plt.xlabel('Longitude')
    plt.ylabel('Elevation (m)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 6: Elevation values over iterations
    plt.subplot(2, 3, 6)
    plt.title('Elevation Values Over Iterations')
    plt.plot(range(1, len(y_sample)+1), y_sample, 'bo-', label='Sampled elevations')
    plt.axhline(current_best, color='r', linestyle='--', label=f'Current best: {current_best:.1f}m')
    plt.xlabel('Iteration')
    plt.ylabel('Elevation (m)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Sampled at: lon={xt[0]:.4f}, lat={xt[1]:.4f}")
    print(f"Elevation: {y_new:.2f}m")
    print(f"Current best: {current_best:.2f}m")

    
print('\n' + '='*60)
print('OPTIMIZATION COMPLETE!')
print('='*60)
best_idx = np.argmax(y_sample)
print(f'Best location: lon={X_sample[best_idx][0]:.4f}, lat={X_sample[best_idx][1]:.4f}')
print(f'Maximum elevation found: {np.max(y_sample):.2f} meters')
print(f'\nAll sampled points:')
for i, (x, y_val) in enumerate(zip(X_sample, y_sample)):
    marker = ' ← BEST' if i == best_idx else ''
    print(f'  {i+1:2d}. lon={x[0]:7.4f}, lat={x[1]:7.4f} → {y_val:7.2f}m{marker}')

Candidate grid shape: (2500, 2) (each point is [lon, lat])


TypeError: 'list' object is not callable